[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/00_question_orientation.ipynb)

# R1-Q2 Week 1 — Question orientation and pre-commit

This notebook does WGCNA-specific preparation for R1-Q2 and produces the consolidated pre-commit file you submit at the end of Week 1.

By the end of this notebook you will have:

- Filtered shoot and root expression matrices ready for WGCNA in Week 2 (`shoot_filtered.parquet`, `root_filtered.parquet`).
- A written pre-commit covering five decision areas: network-construction choices, hub-identification choices, the known-regulators reference set, the statistical framework for the hub-vs-known comparison, and the background gene set.
- A consolidated `precommit.json` written to your output directory, ready as the Week 1 EQ#1 deliverable.

In [ ]:
# Install irilab2026.
# Variants below — the active line is the one that's uncommented.
# Pin to a tagged release:    pip install irilab2026==0.2.0
# Latest tagged release:      pip install irilab2026
# Editable, from a clone:     pip install -e .
!pip install --upgrade --force-reinstall --no-deps irilab2026

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load and inspect

In this section you'll load the AtGenExpress expression data and its
sample metadata, combine the per-stress tables into a single expression
matrix, and split that matrix into a shoot table and a root table. By
the end of this section you'll have two per-tissue matrices ready for
the WGCNA-specific filtering step in Section 3.

The rationale-level `r1_orientation.ipynb` walked through the
AtGenExpress structure (eight stresses + control, shoot and root tissue,
time course up to 24 hours) and the three caveats that apply across all
R1 questions. Those caveats matter just as much for R1-Q2 as they did
for R1-Q1; refresh on them in the orientation notebook if you haven't
worked through it recently.

In [ ]:
# Load the AtGenExpress expression data and the sample-level metadata.
# Both functions hit the cache on subsequent runs in the same session.
data = iri.load_atgenexpress()
metadata = iri.atgenexpress_metadata()

print(f"Loaded {len(data)} stress conditions: {sorted(data.keys())}")
print(f"Metadata: {metadata.shape[0]} samples × {metadata.shape[1]} columns\n")
metadata.head()

In the above cell `data` is a dictionary with one DataFrame per stress (probes by samples).
The per-stress tables share the same probe index but have different
samples in their columns, so we can concatenate them column-wise to get
a single 22,810-probe-by-248-sample expression matrix.

Once we have that combined matrix, the metadata's `tissue` column tells
us which samples are shoot and which are root. We index into the matrix
to produce a shoot-only table and a root-only table.

In [ ]:
import pandas as pd

# Combine the per-stress matrices into a single probes × samples table.
# All AtGenExpress samples were measured on the same Affymetrix ATH1
# array, so the probe set is identical across stresses. We use an inner
# join to be explicit about that — any probe missing from any stress
# would be dropped, but in practice none are missing.
full = pd.concat(data.values(), axis=1, join="inner")

# Use the metadata's tissue column to identify the GSMs in each tissue,
# then index the combined matrix.
shoot_gsms = metadata.index[metadata["tissue"] == "shoot"]
root_gsms = metadata.index[metadata["tissue"] == "root"]
shoot = full[shoot_gsms]
root = full[root_gsms]

print(f"Combined:  {full.shape[0]:>5} probes × {full.shape[1]:>3} samples")
print(f"Shoot:     {shoot.shape[0]:>5} probes × {shoot.shape[1]:>3} samples")
print(f"Root:      {root.shape[0]:>5} probes × {root.shape[1]:>3} samples")

The shapes above carry a methodological warning worth absorbing before
Section 3. For a co-expression network, **samples are observations**
and **genes are features**. Each per-tissue table has on the order of
a hundred-plus samples for estimating relationships across 22,810
genes. That ratio — far more features than observations — is the same
setting that shaped Rationale 1's approach in `r1_orientation.ipynb`
and R1-Q1. WGCNA handles it through soft thresholding and module
detection, but the underlying constraint is unchanged: modeling
choices can fit noise as easily as signal. The pre-commits in
Sections 4–6 are how you keep yourself honest about that.

A three-digit sample count sounds comfortable, but it isn't — not when
the feature count is nearly two orders of magnitude larger.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

ax_shoot.hist(shoot.values.flatten(), bins=80, color="steelblue", log=True)
ax_shoot.set_title(f"Shoot expression values (n={shoot.shape[1]} samples)")
ax_shoot.set_xlabel("MAS5-normalized intensity")
ax_shoot.set_ylabel("Probe-sample count (log scale)")

ax_root.hist(root.values.flatten(), bins=80, color="seagreen", log=True)
ax_root.set_title(f"Root expression values (n={root.shape[1]} samples)")
ax_root.set_xlabel("MAS5-normalized intensity")

plt.tight_layout()
plt.show()

print("\nValue statistics:")
print(f"  shoot:  min={shoot.values.min():>7.1f}   median={np.median(shoot.values):>6.1f}   max={shoot.values.max():>7.1f}")
print(f"  root:   min={root.values.min():>7.1f}   median={np.median(root.values):>6.1f}   max={root.values.max():>7.1f}")

The distributions are right-skewed with a long tail — characteristic of
linear-scale MAS5-normalized microarray intensities. Before filtering on
variance in Section 3, you'll log-transform these values so that
high-expression genes don't dominate the variance calculation on
absolute-magnitude grounds alone. The data shape you've just seen is
the input; Section 3 turns it into the input WGCNA actually expects.